In [11]:
import os
from typing import Callable, Dict, Any
import csv
import torch
import torch.nn as nn
from torchvision.models import (
    resnet18,
    efficientnet_v2_s,
    ResNet18_Weights,
    EfficientNet_V2_S_Weights,
)
from calflops import calculate_flops
import loralib as lora


# =========================================================
# CONFIG
# =========================================================

NUM_CLASSES = 101  # Food101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# HELPERS
# =========================================================

def freeze_all_params(model: nn.Module) -> None:
    for p in model.parameters():
        p.requires_grad = False


def enable_bitfit(model: nn.Module) -> None:
    freeze_all_params(model)
    for name, p in model.named_parameters():
        if "bias" in name:
            p.requires_grad = True


def enable_batchnorm_tuning(model: nn.Module) -> None:
    freeze_all_params(model)
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            if m.weight is not None:
                m.weight.requires_grad = True
            if m.bias is not None:
                m.bias.requires_grad = True


def replace_resnet18_classifier(model: nn.Module, num_classes: int) -> nn.Module:
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def replace_effnetv2s_classifier(model: nn.Module, num_classes: int) -> nn.Module:
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


# =========================================================
# LORA HELPERS
# =========================================================

def _to_int(x):
    return x if isinstance(x, int) else x[0]


def _get_module(model, name):
    parts = name.split(".")
    m = model
    for p in parts:
        m = m[int(p)] if p.isdigit() else getattr(m, p)
    return m


def _set_module(model, name, new_module):
    parts = name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = parent[int(p)] if p.isdigit() else getattr(parent, p)

    last = parts[-1]
    if last.isdigit():
        parent[int(last)] = new_module
    else:
        setattr(parent, last, new_module)


# =========================================================
# LORA ARCHITECTURE PATCHES
# =========================================================

def apply_resnet18_lora(model: nn.Module, rank: int = 8, alpha: int = 16) -> nn.Module:
    target_layers = [
        "layer1.0.conv1", "layer1.0.conv2",
        "layer1.1.conv1", "layer1.1.conv2",
        "layer2.0.conv1", "layer2.0.conv2",
        "layer2.1.conv1", "layer2.1.conv2",
        "layer3.0.conv1", "layer3.0.conv2",
        "layer3.1.conv1", "layer3.1.conv2",
        "layer4.0.conv1", "layer4.0.conv2",
        "layer4.1.conv1", "layer4.1.conv2",
    ]

    for name in target_layers:
        module = _get_module(model, name)

        new_layer = lora.Conv2d(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=_to_int(module.kernel_size),
            stride=_to_int(module.stride),
            padding=_to_int(module.padding),
            dilation=_to_int(module.dilation),
            groups=module.groups,
            bias=(module.bias is not None),
            r=rank,
            lora_alpha=alpha,
        )

        new_layer.conv.weight.data.copy_(module.weight.data)
        if module.bias is not None:
            new_layer.conv.bias.data.copy_(module.bias.data)

        _set_module(model, name, new_layer)

    return model


def apply_effnetv2s_lora(
    model: nn.Module,
    rank: int = 8,
    alpha: int = 16
) -> nn.Module:
    """
    Matches the actual training code:
    recursively replace every nn.Conv2d in the full EfficientNetV2-S model
    with lora.Conv2d.
    """
    def to_int(x):
        return x if isinstance(x, int) else x[0]

    def replace_module(parent, name, new_module):
        if name.isdigit():
            parent[int(name)] = new_module
        else:
            setattr(parent, name, new_module)

    def recursive_replace(module_parent):
        for name, module in list(module_parent.named_children()):
            if isinstance(module, nn.Conv2d):
                new_layer = lora.Conv2d(
                    in_channels=module.in_channels,
                    out_channels=module.out_channels,
                    kernel_size=to_int(module.kernel_size),
                    stride=to_int(module.stride),
                    padding=to_int(module.padding),
                    dilation=to_int(module.dilation),
                    groups=module.groups,
                    bias=(module.bias is not None),
                    r=rank,
                    lora_alpha=alpha,
                )

                new_layer.conv.weight.data.copy_(module.weight.data)
                if module.bias is not None:
                    new_layer.conv.bias.data.copy_(module.bias.data)

                replace_module(module_parent, name, new_layer)
            else:
                recursive_replace(module)

    recursive_replace(model)
    return model

# =========================================================
# TSA MODULES / ARCHITECTURES
# =========================================================

class TSA(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden_dim = max(1, channels // reduction)
        self.adapter = nn.Sequential(
            nn.Conv2d(channels, hidden_dim, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, channels, kernel_size=1),
        )

    def forward(self, x):
        return x + self.adapter(x)


class ResNet18_TSA(nn.Module):
    def __init__(self, num_classes=101, reduction=16, weights=ResNet18_Weights.IMAGENET1K_V1):
        super().__init__()

        base = resnet18(weights=weights)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = self._add_adapters(base.layer1, reduction)
        self.layer2 = self._add_adapters(base.layer2, reduction)
        self.layer3 = self._add_adapters(base.layer3, reduction)
        self.layer4 = self._add_adapters(base.layer4, reduction)

        self.avgpool = base.avgpool
        self.fc = nn.Linear(base.fc.in_features, num_classes)

    def _add_adapters(self, layer, reduction):
        new_blocks = []
        for block in layer:
            channels = block.conv2.out_channels
            adapter = TSA(channels, reduction)
            new_blocks.append(nn.Sequential(block, adapter))
        return nn.Sequential(*new_blocks)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class EfficientNetV2S_TSA(nn.Module):
    def __init__(self, num_classes=101, reduction=16,
                 weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1):
        super().__init__()

        base = efficientnet_v2_s(weights=weights)
        self.features = self._add_adapters(base.features, reduction)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(base.classifier[1].in_features, num_classes)

    def _infer_out_channels(self, block):
        # robust channel inference for torchvision EfficientNetV2-S blocks
        for m in reversed(list(block.modules())):
            if isinstance(m, nn.Conv2d):
                return m.out_channels
        return None

    def _add_adapters(self, features, reduction):
        new_layers = []

        for layer in features:
            if isinstance(layer, nn.Sequential):
                blocks = []
                for block in layer:
                    channels = self._infer_out_channels(block)
                    if channels is None:
                        blocks.append(block)
                    else:
                        adapter = TSA(channels, reduction)
                        blocks.append(nn.Sequential(block, adapter))
                new_layers.append(nn.Sequential(*blocks))
            else:
                new_layers.append(layer)

        return nn.Sequential(*new_layers)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# =========================================================
# RESNET18 BUILDERS
# input_shape = (1, 3, 224, 224)
# =========================================================

def build_resnet18_linear_probe(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    freeze_all_params(model)
    replace_resnet18_classifier(model, num_classes)
    for p in model.fc.parameters():
        p.requires_grad = True
    return model


def build_resnet18_bitfit(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    replace_resnet18_classifier(model, num_classes)
    enable_bitfit(model)
    return model


def build_resnet18_batchnorm(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    replace_resnet18_classifier(model, num_classes)
    enable_batchnorm_tuning(model)
    for p in model.fc.parameters():
        p.requires_grad = True
    return model


def build_resnet18_lora(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model = apply_resnet18_lora(model, rank=8, alpha=16)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_resnet18_tsa(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = ResNet18_TSA(
        num_classes=num_classes,
        reduction=16,
        weights=ResNet18_Weights.IMAGENET1K_V1
    )

    for param in model.parameters():
        param.requires_grad = False

    for module in model.modules():
        if isinstance(module, TSA):
            for p in module.parameters():
                p.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    return model


# =========================================================
# EFFICIENTNET V2-S BUILDERS
# input_shape = (1, 3, 384, 384)
# =========================================================

def build_effnetv2s_linear_probe(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    freeze_all_params(model)
    replace_effnetv2s_classifier(model, num_classes)
    for p in model.classifier[1].parameters():
        p.requires_grad = True
    return model


def build_effnetv2s_bitfit(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    replace_effnetv2s_classifier(model, num_classes)
    enable_bitfit(model)
    return model


def build_effnetv2s_batchnorm(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    replace_effnetv2s_classifier(model, num_classes)
    enable_batchnorm_tuning(model)
    for p in model.classifier[1].parameters():
        p.requires_grad = True
    return model


def build_effnetv2s_lora(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    model = apply_effnetv2s_lora(model, rank=8, alpha=16)

    lora.mark_only_lora_as_trainable(model)

    for p in model.classifier.parameters():
        p.requires_grad = True

    return model


def build_effnetv2s_tsa(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = EfficientNetV2S_TSA(
        num_classes=num_classes,
        reduction=16,
        weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )

    for param in model.parameters():
        param.requires_grad = False

    for module in model.modules():
        if isinstance(module, TSA):
            for p in module.parameters():
                p.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    return model


# =========================================================
# MODEL REGISTRY
# =========================================================

MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "resnet18_linear_probe": {
        "builder": build_resnet18_linear_probe,
        "input_shape": (1, 3, 224, 224),
    },
    "resnet18_lora": {
        "builder": build_resnet18_lora,
        "input_shape": (1, 3, 224, 224),
    },
    "resnet18_batchnorm": {
        "builder": build_resnet18_batchnorm,
        "input_shape": (1, 3, 224, 224),
    },
    "resnet18_tsa": {
        "builder": build_resnet18_tsa,
        "input_shape": (1, 3, 224, 224),
    },
    "resnet18_bitfit": {
        "builder": build_resnet18_bitfit,
        "input_shape": (1, 3, 224, 224),
    },

    "effnetv2s_linear_probe": {
        "builder": build_effnetv2s_linear_probe,
        "input_shape": (1, 3, 384, 384),
    },
    "effnetv2s_lora": {
        "builder": build_effnetv2s_lora,
        "input_shape": (1, 3, 384, 384),
    },
    "effnetv2s_batchnorm": {
        "builder": build_effnetv2s_batchnorm,
        "input_shape": (1, 3, 384, 384),
    },
    "effnetv2s_tsa": {
        "builder": build_effnetv2s_tsa,
        "input_shape": (1, 3, 384, 384),
    },
    "effnetv2s_bitfit": {
        "builder": build_effnetv2s_bitfit,
        "input_shape": (1, 3, 384, 384),
    },
}


# =========================================================
# CHECKPOINT LOADING
# =========================================================

def extract_state_dict(checkpoint: Any) -> Dict[str, torch.Tensor]:
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    cleaned = {}
    for k, v in state_dict.items():
        cleaned[k.replace("module.", "")] = v
    return cleaned


def load_model_from_registry(model_key: str, checkpoint_path: str, strict: bool = True) -> nn.Module:
    entry = MODEL_REGISTRY[model_key]
    builder: Callable[[], nn.Module] = entry["builder"]
    model = builder()

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    state_dict = extract_state_dict(checkpoint)
    model.load_state_dict(state_dict, strict=strict)

    model.to(DEVICE)
    model.eval()
    return model


# =========================================================
# FLOPS
# =========================================================

def compute_flops_for_model(
    model_key: str,
    checkpoint_path: str,
    strict: bool = True,
    detailed: bool = False,
) -> Dict[str, Any]:
    entry = MODEL_REGISTRY[model_key]
    input_shape = entry["input_shape"]

    model = load_model_from_registry(model_key, checkpoint_path, strict=strict)

    flops, macs, params = calculate_flops(
        model=model,
        input_shape=input_shape,
        print_results=False,
        print_detailed=detailed,
    )

    return {
        "model_key": model_key,
        "checkpoint_path": checkpoint_path,
        "device": DEVICE,
        "input_shape": input_shape,
        "FLOPs": flops,
        "MACs": macs,
        "Params": params,
    }


def compute_all_flops(
    checkpoint_map: Dict[str, str],
    strict: bool = True,
    detailed: bool = False
):
    results = []

    for model_key, checkpoint_path in checkpoint_map.items():
        if model_key not in MODEL_REGISTRY:
            print(f"[SKIP] Unknown model key: {model_key}")
            continue

        if not os.path.exists(checkpoint_path):
            print(f"[SKIP] Missing checkpoint: {checkpoint_path}")
            continue

        try:
            result = compute_flops_for_model(
                model_key=model_key,
                checkpoint_path=checkpoint_path,
                strict=strict,
                detailed=detailed,
            )
            results.append(result)
            print(
                f"[OK] {model_key} | FLOPs={result['FLOPs']} | "
                f"MACs={result['MACs']} | Params={result['Params']}"
            )
        except Exception as e:
            print(f"[FAIL] {model_key}: {e}")

    return results


def print_results_table(results):
    print("\n" + "=" * 120)
    print(f"{'MODEL':30} {'INPUT SHAPE':20} {'FLOPs':20} {'MACs':20} {'Params':20}")
    print("=" * 120)
    for r in results:
        print(
            f"{r['model_key']:30} "
            f"{str(r['input_shape']):20} "
            f"{str(r['FLOPs']):20} "
            f"{str(r['MACs']):20} "
            f"{str(r['Params']):20}"
        )
    print("=" * 120)
    
def save_flops_to_csv(results, filepath="FLOP.csv"):
    if len(results) == 0:
        print("No results to save.")
        return

    # Ensure folder exists
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)

    # Define columns
    fieldnames = [
        "model_key",
        "checkpoint_path",
        "device",
        "input_shape",
        "FLOPs",
        "MACs",
        "Params"
    ]

    with open(filepath, mode="w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for r in results:
            writer.writerow({
                "model_key": r["model_key"],
                "checkpoint_path": r["checkpoint_path"],
                "device": r["device"],
                "input_shape": str(r["input_shape"]),
                "FLOPs": r["FLOPs"],
                "MACs": r["MACs"],
                "Params": r["Params"],
            })

    print(f"FLOPs saved to {filepath}")

# =========================================================
# EDIT THESE PATHS
# =========================================================

checkpoint_map = {
    "resnet18_linear_probe": "D:/Data 586/models/linear_probe_ResNet_best.pth",
    "resnet18_lora": "D:/Data 586/models/lora_full_checkpoint.pth",
    "resnet18_batchnorm": "D:/Data 586/models/batchnorm_finetune_ResNet_best.pth",
    "resnet18_tsa": "D:/Data 586/models/adapter_finetune_resnet18_new_best.pth",
    "resnet18_bitfit": "D:/Data 586/models/ResNet18_BitFit_best.pth",

    "effnetv2s_linear_probe": "D:/Data 586/models/linear_probe_EffecientNet_best_.pth",
    "effnetv2s_lora": "D:/Data 586/models/effcient_lora_best.pth",
    "effnetv2s_batchnorm": "D:/Data 586/models/batchnorm_finetune_EfficientNet_best.pth",
    "effnetv2s_tsa": "D:/Data 586/models/efficientnet_adapter_finetune_effcientnet_tsa_v2_best.pth",
    "effnetv2s_bitfit": "D:/Data 586/models/EfficientNetV2S_BitFit_best.pth",
}


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":
    results = compute_all_flops(checkpoint_map, strict=True, detailed=False)

    print_results_table(results)

    # Save to CSV
    save_flops_to_csv(results, filepath="results/FLOP.csv")

[OK] resnet18_linear_probe | FLOPs=3.64 GFLOPS | MACs=1.81 GMACs | Params=51.81 K
[OK] resnet18_lora | FLOPs=3.64 GFLOPS | MACs=1.81 GMACs | Params=763.56 K
[OK] resnet18_batchnorm | FLOPs=3.64 GFLOPS | MACs=1.81 GMACs | Params=61.41 K
[OK] resnet18_tsa | FLOPs=3.66 GFLOPS | MACs=1.83 GMACs | Params=140.89 K
[OK] resnet18_bitfit | FLOPs=3.64 GFLOPS | MACs=1.81 GMACs | Params=4.9 K
[OK] effnetv2s_linear_probe | FLOPs=16.84 GFLOPS | MACs=8.36 GMACs | Params=129.38 K
[OK] effnetv2s_lora | FLOPs=16.84 GFLOPS | MACs=8.36 GMACs | Params=3.94 M
[OK] effnetv2s_batchnorm | FLOPs=16.84 GFLOPS | MACs=8.36 GMACs | Params=283.25 K
[OK] effnetv2s_tsa | FLOPs=17.03 GFLOPS | MACs=8.46 GMACs | Params=509.83 K
[OK] effnetv2s_bitfit | FLOPs=16.84 GFLOPS | MACs=8.36 GMACs | Params=112.23 K

MODEL                          INPUT SHAPE          FLOPs                MACs                 Params              
resnet18_linear_probe          (1, 3, 224, 224)     3.64 GFLOPS          1.81 GMACs           51.81 K  

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# PATHS
# =========================================================
RESNET_CSV = r"D:\Project_Final\training_scripts\ResNet18\results\ResNet18_metrics.csv"
EFFNET_CSV = r"D:\Project_Final\training_scripts\EfficientNetV2\results\EfficientNetV2s_metrics.csv"

OUTPUT_DIR = r"D:\Project_Final\plots"


# =========================================================
# COLUMN NAMES
# =========================================================
MODEL_COL = "method"
EPOCH_COL = "epoch"
TRAIN_ACC_COL = "train_acc"
TRAINABLE_PARAMS_COL = "trainable_params"
FLOP_COL = "FLOP (GFlops)"


# =========================================================
# NAME MAPPING
# =========================================================
RESNET_NAME_MAP = {
    "linear_probe": "linear_probe",
    "batchnorm_finetune": "batchnorm",
    "adapter_finetune": "tsa",
    "resnet_real_Lora": "lora",
    "resnet18_bitfit": "bitfit",
}

EFFNET_NAME_MAP = {
    "linear_probe": "linear_probe",
    "batchnorm_finetune": "batchnorm",
    "efficientnet_adapter_finetune": "tsa",
    "efficientnet_lora": "lora",
    "efficientnet_bitfit": "bitfit",
}


# =========================================================
# HELPERS
# =========================================================
def make_bubble_sizes(values, min_size=300, max_size=2500):
    values = pd.Series(values).astype(float)
    vmin = values.min()
    vmax = values.max()

    if vmax == vmin:
        return pd.Series([0.5 * (min_size + max_size)] * len(values), index=values.index)

    scaled = (values - vmin) / (vmax - vmin)
    return min_size + scaled * (max_size - min_size)


def load_data(csv_path, name_map):
    df = pd.read_csv(csv_path)

    df[MODEL_COL] = df[MODEL_COL].astype(str).str.strip().replace(name_map)
    df[EPOCH_COL] = pd.to_numeric(df[EPOCH_COL], errors="coerce")

    df = df[df[EPOCH_COL] == 8].copy()

    df[TRAIN_ACC_COL] = pd.to_numeric(df[TRAIN_ACC_COL], errors="coerce")
    df[TRAINABLE_PARAMS_COL] = pd.to_numeric(df[TRAINABLE_PARAMS_COL], errors="coerce")
    df[FLOP_COL] = pd.to_numeric(df[FLOP_COL], errors="coerce")

    df = df[[MODEL_COL, TRAIN_ACC_COL, TRAINABLE_PARAMS_COL, FLOP_COL]].dropna()

    df.columns = ["model", "train_acc", "trainable_params", "flops"]
    return df


def plot_bubble(df, title, save_path):
    bubble_sizes = make_bubble_sizes(df["trainable_params"])

    plt.figure(figsize=(10, 7))
    plt.scatter(
        df["train_acc"],
        df["flops"],
        s=bubble_sizes,
        alpha=0.6
    )

    for _, row in df.iterrows():
        plt.annotate(
            row["model"],
            (row["train_acc"], row["flops"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9
        )

    plt.xlabel("Training Accuracy")
    plt.ylabel("FLOP Score (GFlops)")
    plt.title(title)
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")


# =========================================================
# MAIN
# =========================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load data
resnet_df = load_data(RESNET_CSV, RESNET_NAME_MAP)
effnet_df = load_data(EFFNET_CSV, EFFNET_NAME_MAP)

if resnet_df.empty:
    print("Warning: ResNet data is empty")

if effnet_df.empty:
    print("Warning: EfficientNet data is empty")


# Plot separately
plot_bubble(
    resnet_df,
    "ResNet18: FLOP vs Training Accuracy",
    os.path.join(OUTPUT_DIR, "resnet_bubble_plot.png")
)

plot_bubble(
    effnet_df,
    "EfficientNetV2-S: FLOP vs Training Accuracy",
    os.path.join(OUTPUT_DIR, "effnet_bubble_plot.png")
)

Saved: D:\Project_Final\plots\resnet_bubble_plot.png
Saved: D:\Project_Final\plots\effnet_bubble_plot.png
